# Alexandria Lint/Split Smoke Notebook

Deterministic local walkthrough for Delivery 3. It builds a full active leaf, queues a `split.check` job, runs the worker boundary, and inspects the durable before/after state. The splitter, embedder, and summarizer are local fakes.

In [ ]:
from __future__ import annotations

from uuid import UUID

from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, sessionmaker

from application.app import App
import application.app as app_module
from application.ports import ChildPlan, DocIn, SplitPlan
from domain.entity import Base, Document, Job, Node, Reference, VECTOR_DIMENSIONS
from domain.values import JobKind
from infrastructure.config import IngestSettings, Settings
from infrastructure.repositories.outbox import OutboxRepo
from presentation.worker.app import Worker


def uid(value: int) -> UUID:
    return UUID(f"00000000-0000-0000-0000-{value:012x}")


def vector(*values: float) -> list[float]:
    embedding = [0.0] * VECTOR_DIMENSIONS
    for index, value in enumerate(values):
        embedding[index] = value
    return embedding


class MemoryDb:
    def __init__(self) -> None:
        self.engine = create_engine("sqlite+pysqlite:///:memory:")
        self.factory = sessionmaker(bind=self.engine)

    def sessions(self) -> sessionmaker[Session]:
        return self.factory

    def session(self) -> Session:
        return self.factory()

    def create_all(self) -> None:
        Base.metadata.create_all(self.engine)


class FakeEmbedder:
    async def embed(self, text: str) -> list[float]:
        return vector(float(len(text)))


class FakeSummarizer:
    async def summarize(self, doc: DocIn) -> str:
        return f"summary:{doc.name}"


class DeterministicSplitter:
    async def split(self, node: Node, docs: list[Document]) -> SplitPlan:
        ordered = sorted(docs, key=lambda item: item.name)
        return SplitPlan(
            children=[
                ChildPlan(
                    name="Alpha child",
                    description="Documents about alpha topics.",
                    embedding=vector(0.1),
                    docs=[ordered[0].id, ordered[2].id],
                ),
                ChildPlan(
                    name="Beta child",
                    description="Documents about beta topics.",
                    embedding=vector(0.2),
                    docs=[ordered[1].id],
                ),
            ]
        )


In [ ]:
db = MemoryDb()
app_module.Db = lambda _settings: db
app_module.make_embedder = lambda _provider, _settings: FakeEmbedder()
app_module.make_summarizer = lambda _provider, _settings: FakeSummarizer()

settings = Settings(_env_file=None, ingest=IngestSettings(max_leaf_docs=2))
app = App(settings, splitter=DeterministicSplitter())


In [ ]:
with db.session() as session:
    source = Node(
        id=uid(1),
        name="Source",
        description="Full source leaf.",
        embedding=vector(1.0),
        kind="leaf",
        status="active",
        doc_count=3,
    )
    target = Node(
        id=uid(2),
        name="Target",
        description="Referenced target leaf.",
        embedding=vector(2.0),
        kind="leaf",
        status="active",
    )
    docs = [
        Document(id=uid(10), leaf=source, name="Alpha", summary="A", body="A", embedding=vector(10.0)),
        Document(id=uid(11), leaf=source, name="Beta", summary="B", body="B", embedding=vector(11.0)),
        Document(id=uid(12), leaf=source, name="Gamma", summary="G", body="G", embedding=vector(12.0)),
    ]
    ref = Reference(id=uid(20), from_node=source, to_node=target, distance=0.2, rank=0)
    session.add_all([source, target, *docs, ref])
    session.flush()
    outbox = OutboxRepo(session, settings.queue)
    job_id = await outbox.append(Job(kind=JobKind.SPLIT_CHECK, payload={"node_id": str(source.id)}, key=source.id))
    session.commit()

print("job_id", job_id)


In [ ]:
with db.session() as session:
    source = session.get(Node, uid(1))
    source_docs = session.scalars(select(Document).where(Document.leaf_id == uid(1))).all()
    refs = session.scalars(select(Reference).where(Reference.from_node_id == uid(1))).all()
    print("before", source.kind, source.status, source.doc_count, [doc.name for doc in source_docs], len(refs))


In [ ]:
processed = await Worker(app=app, kind=JobKind.SPLIT_CHECK).batch()
print("processed", processed)


In [ ]:
with db.session() as session:
    source = session.get(Node, uid(1))
    children = session.scalars(select(Node).where(Node.parent_id == uid(1)).order_by(Node.name.asc())).all()
    docs = session.scalars(select(Document).order_by(Document.name.asc())).all()
    refs = session.scalars(select(Reference).where(Reference.from_node_id == uid(1))).all()
    job = session.get(Job, job_id)

    print("source", source.kind, source.status, source.doc_count, source.version)
    print("children", [(child.name, child.doc_count) for child in children])
    print("docs", [(doc.name, str(doc.leaf_id)) for doc in docs])
    print("outgoing_refs", len(refs))
    print("job", job.status, job.done_at is not None)
